# Treinamento RT-DETR-X para Detecção de Lixo Urbano

Este notebook documenta e executa o processo de treinamento do modelo RT-DETR-X (Real-Time Detection Transformer Extra Large).
O objetivo é detectar descarte ilegal de lixo urbano em tempo real, utilizando imagens capturadas por uma câmera de segurança externa Tapo.

## 1. Premissas e Configurações de Treino
O dataset contém 3.345 imagens com uma classe única (Lixo), predominantemente em resolução 640x640.
O lixo é um objeto amorfo com alta variância intra-classe. Sendo assim, a pipeline de treino no `train_rtdetr.py` foi configurada com:
- **Resolução de entrada (imgsz):** 640 (Letterboxing nativo para imagens menores).
- **Otimizador:** AdamW (obrigatório para Transformers).
- **Paciência (Early Stopping):** 100 épocas (aguardar convergência da atenção global).
- **Precisão Mista (AMP):** Ativado para ganho de performance em placas NVIDIA.
- **Data Augmentation:** Mosaic, MixUp, Copy-Paste, Fliplr, e Escalonamento agressivo.
- **RESTRIÇÃO CRÍTICA:** `flipud=0.0`. O lixo responde à gravidade; treinar de cabeça para baixo arruina a inferência.
- **Prevenção de Overfitting e Decoramento de cor:** Random Erasing e alta variação HSV aplicados.

## 2. Inicialização do Treinamento

Abaixo, podemos chamar o script modular de treinamento diretamente do notebook para acompanhar a progressão interativamente ou verificar se a GPU está alocando memória corretamente.

*Nota: Para sessões de treinamento longas, recomenda-se executar o script `.py` diretamente no terminal usando `nohup` ou `tmux` para evitar que a sessão do notebook desconecte.*

In [ ]:
import sys
sys.path.append("../../") # Permite importar modulos da raiz do projeto

from src.models.train_rtdetr import train_rtdetr

# Configurando caminhos (ajuste se necessário)
DATA_YAML = "../../data/dataset_final/data.yaml"
PROJECT_DIR = "../../runs/detect"
RUN_NAME = "rtdetr_garbage_detection"

# Executar treinamento (Comente esta célula caso vá executar no terminal)
# results = train_rtdetr(data_yaml_path=DATA_YAML, project_dir=PROJECT_DIR, name=RUN_NAME)

## 3. Análise dos Resultados (Pós-Treinamento)

Após a conclusão do treinamento, os logs, matrizes de confusão e gráficos de aprendizado (loss/época) são salvos na pasta `runs/detect/rtdetr_garbage_detection`.
Podemos visualizar os resultados de aprendizado diretamente aqui.

In [ ]:
import os
from IPython.display import Image, display

results_dir = f"{PROJECT_DIR}/{RUN_NAME}"

if os.path.exists(results_dir):
    print("Arquivos salvos na pasta de run:")
    print(os.listdir(results_dir))
else:
    print(f"A pasta {results_dir} não existe ainda. Execute o treinamento primeiro.")

### Curvas de Aprendizado (Loss) e F1-Score

Verifique se a loss do modelo caiu consistentemente sem que a validação subisse bruscamente (sinal de overfitting).

In [ ]:
# Visualizando resultados de predição/aprendizado
try:
    display(Image(filename=f"{results_dir}/results.png"))
except FileNotFoundError:
    print("results.png não encontrado. O treinamento provavelmente não terminou.")

In [ ]:
# Visualizando matriz de confusão e Curva F1
try:
    print("Matriz de Confusão:")
    display(Image(filename=f"{results_dir}/confusion_matrix.png", width=600))
    print("Curva F1:")
    display(Image(filename=f"{results_dir}/F1_curve.png", width=600))
except FileNotFoundError:
    print("Gráficos de análise não encontrados.")